# **Synthetic Dataset for Student Dropout Prediction**
***by** Ayala Cassiani Olando José (17542) **and** Hernandez Carrillo Sheila Daniela (18038)*

## Objective
This project generates a synthetic dataset that simulates a university student dropout scenario.

The dataset includes:
- Student identification data
- Demographic variables
- Academic variables
- Financial variables
- A binary target variable: **dropout (Yes/No)**

The academic variables are aligned with the University of the Coast grading scale, which uses a **0.0 to 5.0** system in undergraduate programs, with **3.0** as the passing threshold.

Additionally, the dataset includes:
- Missing values
- Outliers
- Categorical variables

This dataset is intended for educational and machine learning purposes.

## Importing required libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Setting seed for reproducibility
np.random.seed(42)

## Data Generation

In this section, we generate synthetic data for 500 students.

The dataset includes:
- Student identification data
- Demographic data
- Academic data
- Financial data

The academic grades are generated using the University of the Coast grading scale, which ranges from **0.0 to 5.0** in undergraduate programs.

In [ ]:
n = 500

# Student identification
first_names = [
    "Daniel", "Maria", "Juan", "Sofia", "Andres", "Valentina",
    "Camilo", "Laura", "Santiago", "Paula", "Carlos", "Camila",
    "Sebastian", "Isabella", "Mateo", "Natalia", "David", "Gabriela"
]

last_names = [
    "Gomez", "Rodriguez", "Perez", "Torres", "Gonzalez", "Ramirez",
    "Lopez", "Martinez", "Hernandez", "Vargas", "Castro", "Moreno"
]

student_name = [
    f"{np.random.choice(first_names)} {np.random.choice(last_names)}"
    for _ in range(n)
]

student_id = [f"CUC-{1000+i}" for i in range(n)]

# Demographic variables
age = np.random.randint(16, 30, n)
gender = np.random.choice(['Male', 'Female', 'Other'], n, p=[0.48, 0.50, 0.02])
origin = np.random.choice(
    ['Barranquilla', 'Soledad', 'Cartagena', 'Santa Marta', 'Monteria', 'Other'],
    n
)

# Academic variables

# University of the Coast scale: 0.0 to 5.0
high_school_avg = np.random.normal(75, 10, n).clip(0, 100)
admission_score = np.random.normal(70, 12, n).clip(0, 100)

first_cut_grade = np.random.normal(3.3, 0.7, n).clip(0, 5)
second_cut_grade = np.random.normal(3.2, 0.7, n).clip(0, 5)
third_cut_grade = np.random.normal(3.4, 0.7, n).clip(0, 5)

first_semester_grade = (
    0.3 * first_cut_grade +
    0.3 * second_cut_grade +
    0.4 * third_cut_grade
).round(2)

# Financial variables
socioeconomic_level = np.random.choice([1, 2, 3, 4, 5, 6], n, p=[0.08, 0.15, 0.25, 0.22, 0.18, 0.12])
scholarship = np.random.choice(['Yes', 'No'], n, p=[0.35, 0.65])
loan = np.random.choice(['Yes', 'No'], n, p=[0.30, 0.70])
financial_aid = np.random.choice(['Yes', 'No'], n, p=[0.25, 0.75])

## Target Variable Creation

The dropout variable is generated using a synthetic risk score based on:
- Low academic performance
- Low socioeconomic level
- Lack of financial support
- Higher student age

The university grade threshold is considered at **3.0**, which is the passing minimum in the University of the Coast grading scale.

In [ ]:
risk = (
    (age > 24).astype(int) * 0.10 +
    (high_school_avg < 60).astype(int) * 0.20 +
    (admission_score < 55).astype(int) * 0.15 +
    (first_semester_grade < 3.0).astype(int) * 0.40 +
    (socioeconomic_level <= 2).astype(int) * 0.10 +
    (scholarship == 'No').astype(int) * 0.05
)

prob_dropout = 1 / (1 + np.exp(-(risk - 0.35)))
dropout = np.where(np.random.rand(n) < prob_dropout, 'Yes', 'No')

## Dataset Construction

All variables are combined into a structured dataset using a pandas DataFrame.

This structure includes student identification, demographic information, academic performance, financial conditions, and the target variable.

In [ ]:
df = pd.DataFrame({
    'student_id': student_id,
    'student_name': student_name,
    'age': age,
    'gender': gender,
    'origin': origin,
    'high_school_avg': high_school_avg.round(2),
    'admission_score': admission_score.round(2),
    'first_cut_grade': first_cut_grade.round(2),
    'second_cut_grade': second_cut_grade.round(2),
    'third_cut_grade': third_cut_grade.round(2),
    'first_semester_grade': first_semester_grade,
    'socioeconomic_level': socioeconomic_level,
    'scholarship': scholarship,
    'loan': loan,
    'financial_aid': financial_aid,
    'dropout': dropout
})

df.head()

## Introducing Missing Values

To simulate real-world data quality issues, missing values are introduced in selected columns.

The student identification fields remain complete because they are essential for traceability.

In [ ]:
for col in ['gender', 'origin', 'high_school_avg', 'loan', 'second_cut_grade']:
    idx = np.random.choice(df.index, size=15, replace=False)
    df.loc[idx, col] = np.nan

df.isnull().sum()

## Introducing Outliers

Outliers are manually added to simulate abnormal values in the dataset.

These extreme values help represent unusual academic or demographic cases that may appear in real data.

In [ ]:
outliers = np.random.choice(df.index, size=8, replace=False)

# Age outliers
age_outliers = [45, 50, 52, 60]
for i, idx in enumerate(outliers[:4]):
    df.loc[idx, 'age'] = age_outliers[i]

# Academic outliers on the 0.0 to 5.0 grading scale
grade_outliers = [0.0, 0.2, 5.6, 6.0]
for i, idx in enumerate(outliers[4:]):
    df.loc[idx, 'first_semester_grade'] = grade_outliers[i]

## Data Visualization

Basic visualizations are created to analyze the dataset distribution and detect patterns.

These plots help validate the presence of categorical variables, missing values, and outliers.

In [ ]:
# Age distribution
plt.figure(figsize=(8, 4))
plt.hist(df['age'].dropna(), bins=15)
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.show()

# First semester grade distribution
plt.figure(figsize=(8, 4))
plt.hist(df['first_semester_grade'].dropna(), bins=15)
plt.title("First Semester Grade Distribution")
plt.xlabel("Grade")
plt.ylabel("Frequency")
plt.show()

# Dropout distribution
plt.figure(figsize=(8, 4))
df['dropout'].value_counts().plot(kind='bar')
plt.title("Dropout Distribution")
plt.xlabel("Dropout")
plt.ylabel("Count")
plt.show()

# Boxplot for outliers
plt.figure(figsize=(8, 4))
plt.boxplot([
    df['age'].dropna(),
    df['first_semester_grade'].dropna()
], tick_labels=['Age', 'First Semester Grade'])
plt.title("Outlier Detection")
plt.ylabel("Values")
plt.show()

## Exporting Dataset

The dataset is exported in both CSV and Excel formats.

In [ ]:
df.to_csv('student_dropout_dataset.csv', index=False)
print("CSV file exported successfully")